# Stage 3: Adversarial Domain Adaptation Fine-Tuning

Following Gupta et al. (ICML 2025, arXiv:2510.12958):
- Load pretrained encoder from Stage 2
- Add domain discriminator: 2-layer MLP predicting sim vs real
- Train with competing objectives:
  1. Classification loss (planet/no-planet) — BCE with pos_weight
  2. Domain loss (sim/real) — discriminator tries to classify domain
  3. Encoder trained to MINIMIZE discriminator accuracy (fool it)
- Forces domain-agnostic representations
- lambda_domain = 0.1 (conservative, following Gupta et al.)

Key fixes from code review:
- Discriminator: detach encoder output (not no_grad) so disc gets gradients
- Batch-size equalization for adversarial loss
- Single forward pass (no duplicate real forward)
- Val data built once outside training loop
- Model classes imported from transformer_model.py (no duplication)

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "0"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Import model from centralized module (no inline class duplication)
from transformer_model import ExoplanetTransformerWithDomain

model = ExoplanetTransformerWithDomain().to(device)
pretrained = torch.load('/kaggle/working/pretrained_stage2.pth')
# Load encoder weights (input_proj, transformer, pool) but not classifier/domain_disc
pretrained_keys = {k: v for k, v in pretrained.items()
                   if not k.startswith('classifier') and not k.startswith('domain_disc')}
model.load_state_dict(pretrained_keys, strict=False)
print('Loaded pretrained encoder weights (excluding classifier + domain_disc).')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Load real data and prepare mixed batches
observations = pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl')
from split import ensure_split
train_stars, val_stars, test_stars = ensure_split(observations)

NUM_FREQS = 8
PERIODS = np.logspace(np.log10(1.0), np.log10(7300.0), NUM_FREQS)
FREQS = 2.0 * np.pi / PERIODS

pos_stars = sorted(set(observations[observations['has_exoplanets'] == 1]['star_name']))
neg_stars = sorted(set(observations[observations['has_exoplanets'] == 0]['star_name']))

def star_to_features_real(star_name):
    star_obs = observations[observations['star_name'] == star_name].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]
    features = []
    for _, row in star_obs.iterrows():
        dt = row['bjd'] - ref_bjd
        pos_enc = []
        for f in FREQS: pos_enc.append(np.sin(f*dt)); pos_enc.append(np.cos(f*dt))
        features.append([row['rv_centered'], row['rv_err'], row['exposure_time'],
                        row['RHKp'], row['Halpha']] + pos_enc)
    return np.array(features, dtype=np.float32)

# Build train/val datasets ONCE (outside training loop)
sim_mean = np.load('/kaggle/working/sim_norm_mean.npy')
sim_std = np.load('/kaggle/working/sim_norm_std.npy')

train_pos = [s for s in train_stars if s in set(pos_stars)]
train_neg = [s for s in train_stars if s in set(neg_stars)]
val_pos = [s for s in val_stars if s in set(pos_stars)]
val_neg = [s for s in val_stars if s in set(neg_stars)]

train_data = [(star_to_features_real(s), 1) for s in train_pos] + [(star_to_features_real(s), 0) for s in train_neg]
train_data = [((seq - sim_mean) / sim_std, label) for seq, label in train_data]

val_data = [(star_to_features_real(s), 1) for s in val_pos] + [(star_to_features_real(s), 0) for s in val_neg]
val_data = [((seq - sim_mean) / sim_std, label) for seq, label in val_data]

print(f'Real training stars: {len(train_data)} ({sum(l for _, l in train_data)} positive)')
print(f'Real val stars: {len(val_data)} ({sum(l for _, l in val_data)} positive)')

# Generate simulated stars for domain mixing
from sim_utils import simulate_dataset
from sim_dataset import star_to_features as sim_feat_fn, collate_stars

print("Generating 5,000 simulated stars for domain mixing...")
sim_stars = simulate_dataset(observations, n_sim_stars=5000, positive_fraction=0.5, seed=seed+2)
sim_data = [(sim_feat_fn(s), int(s['has_exoplanets'])) for _, s in sim_stars]
sim_data = [((seq - sim_mean) / sim_std, label) for seq, label in sim_data]

In [ ]:
# Adversarial training loop
class StarDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0], dtype=torch.float32), torch.tensor(self.data[idx][1], dtype=torch.float32)

batch_size = 32
real_loader = DataLoader(StarDataset(train_data), batch_size=batch_size,
                        shuffle=True, collate_fn=collate_stars, pin_memory=True)
sim_loader = DataLoader(StarDataset(sim_data), batch_size=batch_size,
                       shuffle=True, collate_fn=collate_stars, pin_memory=True)
val_loader = DataLoader(StarDataset(val_data), batch_size=batch_size,
                       shuffle=False, collate_fn=collate_stars)

lambda_domain = 0.1  # Following Gupta et al. ICML 2025
classification_criterion = nn.BCEWithLogitsLoss()
domain_criterion = nn.BCEWithLogitsLoss()

optimizer_main = Adam(
    list(model.input_proj.parameters()) + list(model.transformer.parameters()) +
    list(model.pool.parameters()) + list(model.classifier.parameters()),
    lr=1e-4, weight_decay=5e-3)
optimizer_disc = Adam(model.domain_disc.parameters(), lr=1e-3, weight_decay=1e-4)

warmup_epochs, total_epochs = 3, 50

def lr_lambda(epoch):
    if epoch < warmup_epochs: return (epoch + 1) / warmup_epochs
    return 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs) / (total_epochs - warmup_epochs)))

scheduler = LambdaLR(optimizer_main, lr_lambda)
train_losses, val_aucs = [], []
best_val_auc = 0
best_model_state = None

n_pos = sum(l for _, l in train_data)
n_neg = len(train_data) - n_pos
# Standard pos_weight = n_neg/n_pos (not sqrt) for BCEWithLogitsLoss
pos_weight = torch.tensor([n_neg / max(n_pos, 1)]).to(device)

for epoch in range(total_epochs):
    model.train()
    epoch_loss = 0
    all_probs, all_labels = [], []
    sim_iter = iter(sim_loader)

    for real_padded, real_mask, real_labels in real_loader:
        real_padded, real_mask, real_labels = real_padded.to(device), real_mask.to(device), real_labels.to(device)

        try: sim_padded, sim_mask, _ = next(sim_iter)
        except StopIteration:
            sim_iter = iter(sim_loader); sim_padded, sim_mask, _ = next(sim_iter)
        sim_padded, sim_mask = sim_padded.to(device), sim_mask.to(device)

        # ── Step 1: Train discriminator (freeze encoder, train only domain_disc) ──
        # FIX C2: Detach encoder output so discriminator gets gradients,
        # but encoder parameters are NOT updated in this step.
        optimizer_disc.zero_grad()
        with torch.no_grad():
            z_real = model.encode(real_padded, real_mask)
            z_sim = model.encode(sim_padded, sim_mask)
        # domain_disc forward on detached representations -> disc gets grads
        real_domain_logits = model.domain_disc(z_real.detach())
        sim_domain_logits = model.domain_disc(z_sim.detach())
        real_domain_labels = torch.ones(real_padded.size(0), device=device)
        sim_domain_labels = torch.zeros(sim_padded.size(0), device=device)
        disc_loss = domain_criterion(real_domain_logits, real_domain_labels) + \
                    domain_criterion(sim_domain_logits, sim_domain_labels)
        disc_loss.backward(); optimizer_disc.step()

        # ── Step 2: Train encoder + classifier (with adversarial domain loss) ──
        # FIX: Single forward pass for real data (no duplicate computation)
        optimizer_main.zero_grad()
        class_logits, real_domain_for_enc, _ = model.forward_with_domain(real_padded, real_mask)
        class_loss = F.binary_cross_entropy_with_logits(class_logits, real_labels, pos_weight=pos_weight)

        # Domain adversarial loss: encoder tries to FOOL the discriminator
        # Equalize batch sizes: use min(real_b, sim_b) samples to avoid shape mismatch
        _, sim_domain_for_enc, _ = model.forward_with_domain(sim_padded, sim_mask)
        min_b = min(real_padded.size(0), sim_padded.size(0))
        # Real samples should look like sim (label 0), sim should look like real (label 1)
        fake_real_labels = torch.zeros(min_b, device=device)  # fool disc into saying sim
        fake_sim_labels = torch.ones(min_b, device=device)   # fool disc into saying real
        adv_loss = domain_criterion(real_domain_for_enc[:min_b], fake_real_labels) + \
                   domain_criterion(sim_domain_for_enc[:min_b], fake_sim_labels)
        total_loss = class_loss + lambda_domain * adv_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_main.step()
        epoch_loss += total_loss.item() * real_padded.size(0)
        all_probs.extend(torch.sigmoid(class_logits).detach().cpu().numpy())
        all_labels.extend(real_labels.cpu().numpy())

    scheduler.step()
    train_auc = roc_auc_score(all_labels, all_probs)

    # Validation (val data already built outside loop)
    model.eval()
    val_probs, val_labels = [], []
    with torch.no_grad():
        for padded, mask, labels in val_loader:
            padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
            logits = model(padded, mask)  # forward() returns logits only
            val_probs.extend(torch.sigmoid(logits).cpu().numpy())
            val_labels.extend(labels.numpy())
    val_auc = roc_auc_score(val_labels, val_probs)

    train_losses.append(epoch_loss / len(train_data))
    val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch+1}/{total_epochs} | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}')

print(f'\nBest val AUC: {best_val_auc:.4f}')
torch.save(best_model_state, '/kaggle/working/finetuned_adversarial.pth')
print('Saved adversarial fine-tuned model.')

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(1, 1, figsize=(8, 5))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_aucs, label='Val AUC')
ax1.set_title('Adversarial Fine-Tuning: Loss and Val AUC')
ax1.legend(); ax1.grid(True, alpha=0.3)

plt.show()